# Khánh — LightGBM và feature extension

Nguồn: study Gate 7 ngày 07/10, artifact LightGBM lịch sử của Khánh; sửa tổ chức/đối chiếu ngày 11/10.

**Hai luồng riêng:** Gate 7 so sánh 15/19 feature trên validation; test lịch sử thuộc model 15 feature có 457 cây. Không ghép điểm test 15 feature với feature/model 19.


In [ ]:
from pathlib import Path
import json
import sys
ROOT = Path.cwd()
if not (ROOT / "src/steel").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/steel").is_dir():
    raise FileNotFoundError("Mở notebook từ repo hoặc notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image


In [ ]:
STUDY = ROOT / "reports/steel/khanh_gate7_2026-10-11"
display(pd.read_csv(STUDY / "validation_comparison.csv"))
display(Image(filename=str(STUDY / "train_correlations_extended_19.png")))


## Ý nghĩa feature

- Ramp: ΔkWh giữa t và t−15 phút, không phải công suất tức thời.
- Apparent-energy proxy: từ điện năng tích phân; không khẳng định công suất biểu kiến chính xác.
- Q/P chỉ tính khi Usage > 5: tránh mẫu số nhỏ, không xác nhận máy đang chạy.
- PF < 90% là cờ nghiên cứu, không phải điều khoản phạt.
- Sensor tại t phải có sẵn khi phát dự báo. Correlation không chứng minh causality/leakage.


In [ ]:
VERIFIED = ROOT / "reports/steel/khanh_lightgbm_verified_2026-10-11/verification.json"
verified = json.loads(VERIFIED.read_text(encoding="utf-8"))
print("Artifact:", verified["feature_count"], "features;", verified["tree_count"], "trees")
print("Đối chiếu số cũ:", verified["recorded_metric_differences"])
display(pd.Series(verified["test"]))


## Tái lập bằng CLI

Cài `requirements-khanh.txt` trước.

- `python -m scripts.run_feature_extension --output-dir reports/steel/local_gate7_01`
- `python -m scripts.check_khanh_lightgbm --output-dir reports/steel/local_lightgbm_01`

Đường dẫn phải mới. Gate 7 không mở calibration/test. Lệnh kiểm artifact là đánh giá hồi cứu, không fit lại model. Policy LightGBM cũ sai hash giữ để inspector chỉ ra lỗi; không được bật trong HGB demo.
